In [1]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from underthesea import sent_tokenize
import pandas as pd
import numpy as np
from rouge_score import rouge_scorer
from bert_score import score as bert_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

class PhoBERTSUM(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(encoder.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        sent_emb = outputs.last_hidden_state.mean(dim=1)
        return self.classifier(sent_emb)

def split_sentences(text):
    if not isinstance(text, str):
        return []
    sentences = sent_tokenize(text)
    return [s.strip() for s in sentences if len(s.strip()) > 10]

def get_top_k_by_ratio(num_sentences, ratio=0.5):
    return max(1, int(num_sentences * ratio))

Device: cuda


In [2]:
MODEL_PATH = r"E:\Project_NguyenMinhVu_2211110063\Source\ai\Model Train\Model_TX\vubert_summary_model.pth"

print(f"Đang load model từ: {MODEL_PATH}")

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

encoder = AutoModel.from_pretrained("vinai/phobert-base").to(DEVICE)
model = PhoBERTSUM(encoder).to(DEVICE)

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

Đang load model từ: E:\Project_NguyenMinhVu_2211110063\Source\ai\Model Train\Model_TX\vubert_summary_model.pth


c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
C:\Users\minhv\AppData\Local\Temp\ipykernel_8776\2076137244.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will n

PhoBERTSUM(
  (encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (position_embeddings): Embedding(258, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNor

In [3]:
@torch.no_grad()
def extractive_summary(text, ratio=0.5, max_len=128):
    sentences = split_sentences(text)
    n = len(sentences)

    if n == 0:
        return ""

    if n == 1:
        return sentences[0]

    top_k = get_top_k_by_ratio(n, ratio)

    encoded = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )
    
    input_ids = encoded["input_ids"].to(DEVICE)
    attention_mask = encoded["attention_mask"].to(DEVICE)
    
    scores = model(input_ids, attention_mask)
    scores = scores.squeeze().cpu()

    top_idx = torch.topk(scores, min(top_k, n)).indices.tolist()
    
    if not isinstance(top_idx, list):
        top_idx = [top_idx]

    top_idx.sort()
    summary = " ".join(sentences[i] for i in top_idx)
    return summary

In [4]:
df = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\trichxuattest.xlsx")

df["sum"] = ""  

for i in range(len(df)):
    content = df.iloc[i]["content"]
    summary = extractive_summary(content, ratio=0.6)
    df.iloc[i, df.columns.get_loc("sum")] = summary

df.to_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\trichxuattest_sum.xlsx")

In [5]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore
from underthesea import word_tokenize
import torch

# ---------------------------
# Tiền xử lý tiếng Việt
# ---------------------------
def preprocess_vi(text):
    text = text.strip()
    text = text.replace("\n", " ")
    tokens = word_tokenize(text, format="text")
    return tokens


# ---------------------------
# Tính ROUGE
# ---------------------------
def compute_rouge(candidate, reference):
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rougeL'],
        use_stemmer=False
    )

    scores = scorer.score(reference, candidate)

    rouge_results = {
        "rouge1_precision": scores['rouge1'].precision,
        "rouge1_recall": scores['rouge1'].recall,
        "rouge1_f1": scores['rouge1'].fmeasure,
        "rougeL_precision": scores['rougeL'].precision,
        "rougeL_recall": scores['rougeL'].recall,
        "rougeL_f1": scores['rougeL'].fmeasure,
    }

    return rouge_results


# ---------------------------
# Tính BERTScore (PhoBERT)
# ---------------------------
from bert_score import score
import torch

def compute_bertscore(candidate, reference):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    P, R, F1 = score(
        [candidate],
        [reference],
        lang="vi",          # QUAN TRỌNG
        model_type=None,    # KHÔNG ghi vinai/phobert-base
        device=device
    )

    return {
        "bertscore_precision": P.item(),
        "bertscore_recall": R.item(),
        "bertscore_f1": F1.item()
    }



# ---------------------------
# Hàm tổng hợp
# ---------------------------
def evaluate_summary(candidate, reference):
    candidate_proc = preprocess_vi(candidate)
    reference_proc = preprocess_vi(reference)

    candidate_text = " ".join(candidate_proc)
    reference_text = " ".join(reference_proc)

    results = {}
    results.update(compute_rouge(candidate_text, reference_text))
    results.update(compute_bertscore(candidate_text, reference_text))

    return results


import pandas as pd
import numpy as np

df = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\trichxuattest_sum.xlsx")
# Danh sách cột thang đo
metric_cols = [
    "rouge1_precision", "rouge1_recall", "rouge1_f1",
    "rougeL_precision", "rougeL_recall", "rougeL_f1",
    "bertscore_precision", "bertscore_recall", "bertscore_f1"
]

# Nếu chưa có cột thì tạo
for col in metric_cols:
    if col not in df.columns:
        df[col] = np.nan

# =========================
# TÍNH ĐIỂM TỪNG ROW
# =========================
for idx, row in df.iterrows():
    content = row.get("content", "")
    summary = row.get("sum", "")

    # Bỏ qua nếu thiếu dữ liệu
    if not isinstance(content, str) or not isinstance(summary, str):
        continue
    if content.strip() == "" or summary.strip() == "":
        continue

    try:
        scores = evaluate_summary(summary, content)

        for k, v in scores.items():
            df.at[idx, k] = float(v)

    except Exception as e:
        print(f"⚠️ Lỗi tại dòng {idx}: {e}")
        continue
    
df.to_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\trichxuattest_score.xlsx", index=False)


c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [6]:
df = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\trichxuattest_score.xlsx")
df.describe()

,Unnamed: 0,summary,grade,rouge1_precision,rouge1_recall,rouge1_f1,rougeL_precision,rougeL_recall,rougeL_f1,bertscore_precision,bertscore_recall,bertscore_f1
count,5.000000,0.0,5.000000,5.0,5.000000,5.000000,5.0,5.000000,5.000000,5.000000,5.000000,5.000000
mean,2.000000,NaN,3.000000,1.0,0.667021,0.798717,1.0,0.667021,0.798717,0.911868,0.902868,0.907209
std,1.581139,NaN,1.581139,0.0,0.066266,0.048411,0.0,0.066266,0.048411,0.031948,0.036836,0.032296
min,0.000000,NaN,1.000000,1.0,0.571429,0.727273,1.0,0.571429,0.727273,0.861649,0.865693,0.865296
25%,1.000000,NaN,2.000000,1.0,0.640244,0.780669,1.0,0.640244,0.780669,0.914240,0.868974,0.891170
50%,2.000000,NaN,3.000000,1.0,0.672012,0.803836,1.0,0.672012,0.803836,0.914558,0.911903,0.913229
75%,3.000000,NaN,4.000000,1.0,0.705765,0.827506,1.0,0.705765,0.827506,0.918191,0.912776,0.913507
max,4.000000,NaN,5.000000,1.0,0.745658,0.854300,1.0,0.745658,0.854300,0.950700,0.954993,0.952842


In [2]:
"""
Batch Extractive Summarization — All-in-one Script
===================================================
Model  : PhoBERTSUM (fine-tuned)
Input  : E:\\Project_NguyenMinhVu_2211110063\\Source\\datasets\\dataset abstractive\\Data_test_1000.xlsx
Output : E:\\Project_NguyenMinhVu_2211110063\\Source\\datasets\\dataset abstractive\\Data_test_1000_extract.xlsx

Chạy nhanh:
    python batch_extract.py
    python batch_extract.py --ratio 0.4
    python batch_extract.py --resume
    python batch_extract.py --content-col "noi_dung" --out-col "tom_tat"
"""

# ===========================================================
# IMPORTS
# ===========================================================
import argparse
import sys
import time
from pathlib import Path
from typing import Optional

import torch
import torch.nn as nn
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from nltk.tokenize import sent_tokenize

# ===========================================================
# ▶▶ CẤU HÌNH — chỉnh tại đây nếu không dùng CLI ◀◀
# ===========================================================
INPUT_PATH   = r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\Data_test_1000.xlsx"
OUTPUT_PATH  = r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\Data_test_1000_extract.xlsx"
MODEL_PATH   = r"E:\Project_NguyenMinhVu_2211110063\Source\ai\Model Train\Model_TX_ver2\final_best_model.pt"
ENCODER_NAME = "vinai/phobert-base"

COL_CONTENT  = "content"      # cột văn bản gốc
COL_OUT      = "summary_end"  # cột kết quả — ghi vào cột có sẵn trong file

DEFAULT_RATIO    = 0.5   # tỉ lệ câu giữ lại (0.0 – 1.0)
MAX_LEN          = 128   # max token mỗi câu khi encode
CHECKPOINT_EVERY = 50    # lưu checkpoint sau mỗi N dòng

# ===========================================================
# MODEL DEFINITION
# ===========================================================

class PhoBERTSUM(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder    = encoder
        # Sequential để khớp với key 'classifier.1.*' được lưu lúc train
        self.classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(encoder.config.hidden_size, 1)
        )

    def forward(self, input_ids, attention_mask):
        outputs  = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sent_emb = outputs.last_hidden_state.mean(dim=1)
        return self.classifier(sent_emb)


# ===========================================================
# EXTRACTOR AGENT
# ===========================================================

class ExtractorAgent:
    """PhoBERTSUM extractive summarizer."""

    def __init__(
        self,
        model_path:   str = MODEL_PATH,
        encoder_name: str = ENCODER_NAME,
        max_len:      int = MAX_LEN,
    ):
        self.model_path   = model_path
        self.encoder_name = encoder_name
        self.max_len      = max_len
        self.device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self._load_model()

    # ── model loading ────────────────────────────────────────

    def _load_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.encoder_name, use_fast=False)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        encoder    = AutoModel.from_pretrained(self.encoder_name).to(self.device)
        self.model = PhoBERTSUM(encoder).to(self.device)

        checkpoint = torch.load(self.model_path, map_location=self.device)
        self.model.load_state_dict(checkpoint["model_state"])
        self.model.eval()

    # ── utilities ────────────────────────────────────────────

    def _split_sentences(self, text: str) -> list:
        if not isinstance(text, str):
            return []
        return [s.strip() for s in sent_tokenize(text) if len(s.strip()) > 10]

    # ── core extractive logic ─────────────────────────────────

    @torch.no_grad()
    def extract(self, text: str, ratio: float = 0.5) -> str:
        sentences = self._split_sentences(text)
        n         = len(sentences)

        if n == 0:
            return ""
        if n == 1:
            return sentences[0]

        top_k   = max(1, int(n * ratio))
        encoded = self.tokenizer(
            sentences,
            padding=True,
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )

        input_ids      = encoded["input_ids"].to(self.device)
        attention_mask = encoded["attention_mask"].to(self.device)

        scores  = self.model(input_ids=input_ids, attention_mask=attention_mask)
        scores  = scores.squeeze().cpu()

        top_idx = torch.topk(scores, min(top_k, n)).indices.tolist()
        if not isinstance(top_idx, list):
            top_idx = [top_idx]
        top_idx.sort()

        return " ".join(sentences[i] for i in top_idx).strip()

    # ── public API ───────────────────────────────────────────

    def run(self, text: str, ratio: float = 0.5) -> str:
        try:
            return self.extract(text, ratio=ratio)
        except Exception as e:
            return f"[Error] {e}"


# ===========================================================
# BATCH RUNNER
# ===========================================================

def parse_args():
    p = argparse.ArgumentParser(description="Batch extractive summarization từ Excel")
    p.add_argument("--input",        default=INPUT_PATH)
    p.add_argument("--output",       default=OUTPUT_PATH)
    p.add_argument("--model",        default=MODEL_PATH)
    p.add_argument("--encoder",      default=ENCODER_NAME)
    p.add_argument("--content-col",  default=COL_CONTENT)
    p.add_argument("--out-col",      default=COL_OUT)
    p.add_argument("--ratio",        type=float, default=DEFAULT_RATIO,
                   help="Tỉ lệ câu giữ lại, ví dụ 0.5 = 50%% số câu")
    p.add_argument("--max-len",      type=int, default=MAX_LEN)
    p.add_argument("--checkpoint",   type=int, default=CHECKPOINT_EVERY)
    p.add_argument("--resume",       action="store_true",
                   help="Tiếp tục: bỏ qua dòng đã có kết quả trong file output")
    # Bỏ qua tham số lạ do Jupyter tự thêm (--f=kernel-xxx.json)
    args, _ = p.parse_known_args()
    return args


def load_dataframe(args) -> pd.DataFrame:
    out_path = Path(args.output)
    if args.resume and out_path.exists():
        df = pd.read_excel(args.output, dtype=str)
        print(f"[INFO] Resume từ output: {args.output} ({len(df)} dòng)")
    else:
        df = pd.read_excel(args.input, dtype=str)
        print(f"[INFO] Đọc input: {args.input} ({len(df)} dòng)")
    if args.out_col not in df.columns:
        df[args.out_col] = ""
    return df


def run_batch(args):
    bar = "=" * 62
    print(bar)
    print("  BATCH EXTRACTIVE SUMMARIZATION")
    print(bar)
    print(f"  Input       : {args.input}")
    print(f"  Output      : {args.output}")
    print(f"  Model       : {args.model}")
    print(f"  Encoder     : {args.encoder}")
    print(f"  Content col : {args.content_col}")
    print(f"  Output col  : {args.out_col}")
    print(f"  Ratio       : {args.ratio}  (giữ {int(args.ratio*100)}% số câu)")
    print(f"  Resume      : {args.resume}")
    print(bar)

    print("[INFO] Đang tải model PhoBERTSUM...")
    agent = ExtractorAgent(
        model_path   = args.model,
        encoder_name = args.encoder,
        max_len      = args.max_len,
    )
    print(f"[INFO] Thiết bị: {agent.device}\n")

    df = load_dataframe(args)

    if args.content_col not in df.columns:
        sys.exit(
            f"[ERROR] Cột '{args.content_col}' không tồn tại.\n"
            f"  Các cột hiện có: {list(df.columns)}"
        )

    total     = len(df)
    processed = skipped = errors = 0
    t0        = time.time()

    for idx in range(total):
        row = df.iloc[idx]

        # Resume: bỏ qua dòng đã có kết quả
        if args.resume:
            val = str(row.get(args.out_col, "")).strip()
            if val and val.lower() not in ("", "nan"):
                skipped += 1
                continue

        content = str(row.get(args.content_col, "")).strip()
        if not content or content.lower() == "nan":
            df.at[idx, args.out_col] = ""
            continue

        # Tóm tắt trích xuất
        try:
            result = agent.run(content, ratio=args.ratio)
        except Exception as e:
            result = f"[ERROR] {e}"
            errors += 1

        df.at[idx, args.out_col] = result
        processed += 1

        # Progress log
        elapsed = time.time() - t0
        avg_sec = elapsed / processed
        eta_min = avg_sec * (total - skipped - processed) / 60
        print(
            f"  [{skipped + processed:>5}/{total}] row={idx} | "
            f"{len(content)}→{len(result)} ký tự | ETA {eta_min:.1f} phút",
            flush=True,
        )

        # Checkpoint
        if processed % args.checkpoint == 0:
            df.to_excel(args.output, index=False)
            print(f"  ✔ Checkpoint → {args.output}")

    # Lưu cuối
    df.to_excel(args.output, index=False)

    total_min = (time.time() - t0) / 60
    print(f"\n{bar}")
    print("  HOÀN TẤT")
    print(f"  Đã xử lý  : {processed} dòng")
    print(f"  Bỏ qua    : {skipped} dòng")
    print(f"  Lỗi       : {errors} dòng")
    print(f"  Thời gian : {total_min:.1f} phút")
    print(f"  Lưu tại   : {args.output}")
    print(bar)


# ===========================================================
# ENTRY POINT
# ===========================================================
if __name__ == "__main__":
    run_batch(parse_args())

  BATCH EXTRACTIVE SUMMARIZATION
  Input       : E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\Data_test_1000.xlsx
  Output      : E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\Data_test_1000_extract.xlsx
  Model       : E:\Project_NguyenMinhVu_2211110063\Source\ai\Model Train\Model_TX_ver2\final_best_model.pt
  Encoder     : vinai/phobert-base
  Content col : content
  Output col  : summary_end
  Ratio       : 0.5  (giữ 50% số câu)
  Resume      : False
[INFO] Đang tải model PhoBERTSUM...


c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
C:\Users\minhv\AppData\Local\Temp\ipykernel_5900\3658097648.py:94: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will n

[INFO] Thiết bị: cuda

[INFO] Đọc input: E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\Data_test_1000.xlsx (1000 dòng)
  [    1/1000] row=0 | 1469→692 ký tự | ETA 1.6 phút
  [    2/1000] row=1 | 529→281 ký tự | ETA 1.2 phút
  [    3/1000] row=2 | 194→155 ký tự | ETA 0.9 phút
  [    4/1000] row=3 | 1416→935 ký tự | ETA 1.3 phút
  [    5/1000] row=4 | 1249→905 ký tự | ETA 1.3 phút
  [    6/1000] row=5 | 1129→752 ký tự | ETA 1.4 phút
  [    7/1000] row=6 | 1151→601 ký tự | ETA 1.4 phút
  [    8/1000] row=7 | 1245→696 ký tự | ETA 1.4 phút
  [    9/1000] row=8 | 263→163 ký tự | ETA 1.3 phút
  [   10/1000] row=9 | 1679→1189 ký tự | ETA 1.4 phút
  [   11/1000] row=10 | 761→456 ký tự | ETA 1.4 phút
  [   12/1000] row=11 | 1060→681 ký tự | ETA 1.4 phút
  [   13/1000] row=12 | 1654→884 ký tự | ETA 1.4 phút
  [   14/1000] row=13 | 1174→615 ký tự | ETA 1.4 phút
  [   15/1000] row=14 | 1174→615 ký tự | ETA 1.4 phút
  [   16/1000] row=15 | 1633→1072 ký tự | ETA 1.5 phút
  [ 

In [1]:
from rouge_score import rouge_scorer
from bert_score import BERTScorer          # dùng class thay vì function
from underthesea import word_tokenize
import torch
import pandas as pd
import numpy as np

# ---------------------------
# Tiền xử lý tiếng Việt
# chỉ dùng cho ROUGE
# ---------------------------
def preprocess_vi(text: str) -> str:
    text = text.strip().replace("\n", " ")
    return word_tokenize(text, format="text")


# ---------------------------
# Tính ROUGE
# ---------------------------
def compute_rouge(candidate: str, reference: str) -> dict:
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=False
    )
    scores = scorer.score(reference, candidate)
    return {
        "rouge1_precision" : scores['rouge1'].precision,
        "rouge1_recall"    : scores['rouge1'].recall,
        "rouge1_f1"        : scores['rouge1'].fmeasure,
        "rouge2_precision" : scores['rouge2'].precision,
        "rouge2_recall"    : scores['rouge2'].recall,
        "rouge2_f1"        : scores['rouge2'].fmeasure,
        "rougeL_precision" : scores['rougeL'].precision,
        "rougeL_recall"    : scores['rougeL'].recall,
        "rougeL_f1"        : scores['rougeL'].fmeasure,
    }


# ---------------------------
# Truncate để tránh vượt max_position PhoBERT
# ---------------------------
def truncate_text(text: str, max_words: int = 180) -> str:
    words = text.split()
    return " ".join(words[:max_words]) if len(words) > max_words else text


# ---------------------------
# Tính BERTScore — dùng scorer đã load sẵn
# ---------------------------
def compute_bertscore(candidate: str, reference: str, scorer: BERTScorer) -> dict:
    cand_trunc = truncate_text(candidate)
    ref_trunc  = truncate_text(reference)

    P, R, F1 = scorer.score([cand_trunc], [ref_trunc])
    return {
        "bertscore_precision" : P.item(),
        "bertscore_recall"    : R.item(),
        "bertscore_f1"        : F1.item(),
    }


# ---------------------------
# Hàm tổng hợp
# ---------------------------
def evaluate_summary(candidate: str, reference: str, content: str,
                     scorer: BERTScorer) -> dict:
    # ROUGE: so summary_model vs summary_pre
    cand_tok = preprocess_vi(candidate)
    ref_tok  = preprocess_vi(reference)
    results  = compute_rouge(cand_tok, ref_tok)

    # BERTScore: so summary_model vs summary_pre
    results.update(compute_bertscore(candidate, reference, scorer))

    # BERTScore: so summary_model vs content
    bs_content = compute_bertscore(candidate, content, scorer)
    results["bertscore_vs_content_precision"] = bs_content["bertscore_precision"]
    results["bertscore_vs_content_recall"]    = bs_content["bertscore_recall"]
    results["bertscore_vs_content_f1"]        = bs_content["bertscore_f1"]

    return results


# =========================
# LOAD MODEL MỘT LẦN DUY NHẤT
# =========================
print("[INFO] Đang tải PhoBERT cho BERTScore...")
bert_scorer = BERTScorer(
    model_type = "vinai/phobert-base",
    num_layers = 8,
    lang       = "vi",
    device     = "cpu",
)
print("[INFO] Tải xong.\n")


# =========================
# BATCH EVALUATION
# =========================
data = pd.read_excel(
    r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\Data_test_1000_extract.xlsx"
)

df = data[:50].copy()

REF_COL     = "summary_pre"
CAND_COL    = "summary_model"
CONTENT_COL = "content"

metric_cols = [
    "rouge1_precision", "rouge1_recall", "rouge1_f1",
    "rouge2_precision", "rouge2_recall", "rouge2_f1",
    "rougeL_precision", "rougeL_recall", "rougeL_f1",
    "bertscore_precision", "bertscore_recall", "bertscore_f1",
    "bertscore_vs_content_precision", "bertscore_vs_content_recall", "bertscore_vs_content_f1",
]
for col in metric_cols:
    if col not in df.columns:
        df[col] = np.nan

for idx, row in df.iterrows():
    candidate = row.get(CAND_COL,    "")
    reference = row.get(REF_COL,     "")
    content   = row.get(CONTENT_COL, "")

    if not all(isinstance(x, str) for x in [candidate, reference, content]):
        continue
    if any(x.strip() == "" for x in [candidate, reference, content]):
        continue

    try:
        scores = evaluate_summary(candidate, reference, content, bert_scorer)
        for k, v in scores.items():
            df.at[idx, k] = float(v)
        if idx % 10 == 0:
            print(f"[{idx}/{len(df)}] done")
    except Exception as e:
        print(f"⚠️ Lỗi tại dòng {idx}: {e}")
        continue

df.to_excel(
    r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\Data_test_1000_extract_score.xlsx",
    index=False
)

print("\n=== Điểm trung bình ===")
print(df[metric_cols].mean().round(4).to_string())

[INFO] Đang tải PhoBERT cho BERTScore...


c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


[INFO] Tải xong.

[0/50] done
[10/50] done
[20/50] done
[30/50] done
[40/50] done

=== Điểm trung bình ===
rouge1_precision                  0.9029
rouge1_recall                     0.7795
rouge1_f1                         0.8288
rouge2_precision                  0.7887
rouge2_recall                     0.6789
rouge2_f1                         0.7223
rougeL_precision                  0.8001
rougeL_recall                     0.6886
rougeL_f1                         0.7329
bertscore_precision               0.8281
bertscore_recall                  0.7841
bertscore_f1                      0.8043
bertscore_vs_content_precision    0.8775
bertscore_vs_content_recall       0.7523
bertscore_vs_content_f1           0.8089
